# Newton's Method for Eigenvalues and Eigenvectors

This notebook turns the symmetric eigenvalue problem

$$A v = \lambda v$$

into a nonlinear root-finding problem and solves it first with Newton's method, then with Broyden's method.

You are encouraged to **edit the code cells** for your own curiosity. Change the matrix `A`, the starting vector `x0`, the tolerance `tol`, the maximum iteration count, and the number of random trials. If you break something, use **Ctrl+Z** to undo the change or rerun the earlier cells.

The goal is not just to run the method, but to use the notebook as a learning tool: experiment, compare behavior, and observe when each method succeeds or fails.


## 1. Nonlinear system

Let

$$x = \begin{bmatrix} v \\ \lambda \end{bmatrix} \in \mathbb{R}^{n+1},$$

where the first `n` entries are the eigenvector candidate `v` and the last entry is the eigenvalue candidate `lambda`.

The eigenvector is only unique up to scaling, so we add the normalization equation

$$1 = \frac12 \|v\|_2^2.$$

Equivalently,

$$\frac12 \|v\|_2^2 - 1 = 0.$$

Therefore the root-finding map is

$$
f(x) =
\begin{bmatrix}
A v - \lambda v \\
\frac12 v^T v - 1
\end{bmatrix}.
$$

A solution `f(x)=0` gives an eigenpair, with `||v||_2 = sqrt(2)` because of the chosen normalization.


## 2. Jacobian

Differentiate each block with respect to `v` and `lambda`.

The first block is

$$A v - \lambda v = (A - \lambda I)v.$$

Its derivative with respect to `v` is

$$A - \lambda I,$$

and its derivative with respect to `lambda` is

$$-v.$$

The last scalar equation is

$$\frac12 v^T v - 1,$$

whose derivative with respect to `v` is

$$v^T,$$

and whose derivative with respect to `lambda` is `0`.

So the Jacobian is the block matrix

$$
J_f(x) =
\begin{bmatrix}
A - \lambda I & -v \\
v^T & 0
\end{bmatrix}.
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=8, suppress=True)


## 3. Implement `f`, `J`, and Newton's method

Newton's method solves the linear system

$$J_f(x_k)\,s_k = -f(x_k)$$

and updates

$$x_{k+1} = x_k + s_k.$$

The implementation below uses `np.linalg.solve`, not an explicit inverse. That is the numerically preferred way to compute the Newton step. In short: computers usually prefer solving a linear system over forming a matrix inverse.


In [ ]:
def split_x(x):
    """Return v and lambda from x = [v; lambda]."""
    return x[:-1], x[-1]


def f_eigenpair(x, A):
    """Nonlinear eigenpair system f(x)=0."""
    v, lam = split_x(x)
    top = A @ v - lam * v
    bottom = 0.5 * (v @ v) - 1.0
    return np.concatenate([top, [bottom]])


def J_eigenpair(x, A):
    """Block Jacobian of f(x)."""
    v, lam = split_x(x)
    n = A.shape[0]
    J = np.zeros((n + 1, n + 1))
    J[:n, :n] = A - lam * np.eye(n)
    J[:n, n] = -v
    J[n, :n] = v
    return J


def newton_eigenpair(A, x0, tol=1e-12, max_iter=30, verbose=True):
    """Newton's method for f(x)=0.

    Parameters
    ----------
    A : square NumPy array
        Symmetric matrix.
    x0 : NumPy array of length n+1
        Initial guess [v0; lambda0].
    tol : float
        Stop when ||f(x)||_2 < tol.
    max_iter : int
        Maximum number of Newton steps.
    verbose : bool
        Print iteration table.
    """
    x = np.array(x0, dtype=float).copy()
    residuals = []

    if verbose:
        print(f"{'k':>3} | {'lambda':>15} | {'||f(x)||_2':>14} | {'step norm':>14}")
        print("-" * 58)

    step_norm = np.nan
    for k in range(max_iter + 1):
        fx = f_eigenpair(x, A)
        res = np.linalg.norm(fx)
        residuals.append(res)
        v, lam = split_x(x)

        if verbose:
            step_text = "---" if k == 0 else f"{step_norm:.6e}"
            print(f"{k:3d} | {lam:15.10f} | {res:14.6e} | {step_text:>14}")

        if res < tol:
            return x, np.array(residuals), True

        J = J_eigenpair(x, A)
        try:
            step = np.linalg.solve(J, -fx)
        except np.linalg.LinAlgError:
            if verbose:
                print("Jacobian is singular or numerically singular.")
            return x, np.array(residuals), False

        step_norm = np.linalg.norm(step)
        x = x + step

    return x, np.array(residuals), False


def normalize_sign(v):
    """Flip sign only for easier comparison in printed output."""
    j = np.argmax(np.abs(v))
    return v if v[j] >= 0 else -v


## 4. Required 2 by 2 test matrix

The problem asks us to test

$$
A = \begin{bmatrix}3 & -1 \\ -1 & 3\end{bmatrix}.
$$

Its eigenvalues are `2` and `4`. With the normalization used here, the corresponding eigenvectors are `[1, 1]^T` and `[1, -1]^T`, up to sign.

Run this cell, then change `x0` to see whether Newton's method finds a different eigenpair.


In [ ]:
A2 = np.array([[3.0, -1.0],
               [-1.0, 3.0]])

# Play with this initial guess.
# x0 = [v1_guess, v2_guess, lambda_guess]
x0 = np.array([0.8, 1.3, 1.5])

x_star, residuals, converged = newton_eigenpair(A2, x0, tol=1e-13, max_iter=20)
v_star, lam_star = split_x(x_star)

print("\nConverged:", converged)
print("lambda ≈", lam_star)
print("v ≈", normalize_sign(v_star))
print("||v||_2 ≈", np.linalg.norm(v_star))
print("final residual ≈", residuals[-1])


### Plot the convergence

Newton's method is locally quadratically convergent when the root is regular and the starting point is close enough. On a semilog plot, quadratic convergence often appears as a rapidly steepening curve rather than a straight line.


In [ ]:
plt.figure()
plt.semilogy(residuals, marker="o")
plt.xlabel("Newton iteration k")
plt.ylabel("||f(x_k)||_2")
plt.title("Newton residual history for the 2 by 2 example")
plt.grid(True)
plt.show()


## 5. Random starting guesses with Newton

The same matrix has two eigenpairs. Random starts may converge to either one, and some unlucky starts may encounter a nearly singular Jacobian.

Edit `num_trials`, `seed`, and the random scale below.


In [ ]:
def run_random_trials(A, solver, num_trials=12, seed=7, scale=2.0, tol=1e-12, max_iter=30, **solver_kwargs):
    rng = np.random.default_rng(seed)
    rows = []
    n = A.shape[0]
    for trial in range(num_trials):
        x0 = scale * rng.normal(size=n + 1)
        x_star, residuals, converged = solver(
            A, x0, tol=tol, max_iter=max_iter, verbose=False, **solver_kwargs
        )
        v, lam = split_x(x_star)
        rows.append({
            "trial": trial,
            "converged": converged,
            "iterations": len(residuals) - 1,
            "lambda": lam,
            "v": normalize_sign(v),
            "final_residual": residuals[-1],
        })
    return rows


def print_trial_rows(rows, show_vector=True):
    for r in rows:
        if show_vector:
            print(f"trial {r['trial']:2d}: converged={str(r['converged']):5s}, "
                  f"iters={r['iterations']:2d}, lambda={r['lambda']: .10f}, "
                  f"v={r['v']}, residual={r['final_residual']:.2e}")
        else:
            print(f"trial {r['trial']:2d}: converged={str(r['converged']):5s}, "
                  f"iters={r['iterations']:2d}, lambda={r['lambda']: .10f}, "
                  f"residual={r['final_residual']:.2e}")


newton_rows_A2 = run_random_trials(A2, newton_eigenpair, num_trials=12, seed=4, scale=2.0)
print_trial_rows(newton_rows_A2)


## 6. Estimate the convergence rate

A rough way to diagnose convergence order is to compare consecutive residuals. If

$$e_{k+1} \approx C e_k^p,$$

then

$$p \approx \frac{\log(e_{k+1}/e_k)}{\log(e_k/e_{k-1})}.$$

This estimate is noisy and only meaningful once the iterates are already close to the root.


In [ ]:
def estimate_orders(residuals):
    residuals = np.asarray(residuals, dtype=float)
    orders = []
    for k in range(1, len(residuals) - 1):
        e0, e1, e2 = residuals[k - 1], residuals[k], residuals[k + 1]
        if e0 > 0 and e1 > 0 and e2 > 0 and e0 != e1 and e1 != e2:
            p = np.log(e2 / e1) / np.log(e1 / e0)
            if np.isfinite(p):
                orders.append((k, p))
    return orders

print("Estimated orders from the earlier Newton 2 by 2 run:")
for k, p in estimate_orders(residuals):
    print(f"using residuals around k={k}: p ≈ {p:.3f}")


## 7. A 3 by 3 symmetric example with Newton

Now test at least one other symmetric matrix. The example below is symmetric and has three real eigenvalues.

Try changing entries of `A3`, but keep it symmetric: if you change `A3[i,j]`, also change `A3[j,i]`.


In [ ]:
A3 = np.array([[ 2.0,  1.0,  0.0],
               [ 1.0,  3.0, -1.0],
               [ 0.0, -1.0,  4.0]])

# NumPy's reference answer for comparison.
true_lams, true_vecs = np.linalg.eigh(A3)
print("Reference eigenvalues from np.linalg.eigh:")
print(true_lams)

# Play with this initial guess.
x0_3 = np.array([1.0, 0.5, -0.5, 2.5])

x3_star, residuals3, converged3 = newton_eigenpair(A3, x0_3, tol=1e-13, max_iter=30)
v3_star, lam3_star = split_x(x3_star)

print("\nConverged:", converged3)
print("lambda ≈", lam3_star)
print("v ≈", normalize_sign(v3_star))
print("||v||_2 ≈", np.linalg.norm(v3_star))
print("final residual ≈", residuals3[-1])


In [ ]:
plt.figure()
plt.semilogy(residuals3, marker="o")
plt.xlabel("Newton iteration k")
plt.ylabel("||f(x_k)||_2")
plt.title("Newton residual history for the 3 by 3 example")
plt.grid(True)
plt.show()


## 8. More random trials for the 3 by 3 matrix with Newton

This cell shows which eigenvalue Newton's method finds from several random starting points.


In [ ]:
newton_rows_A3 = run_random_trials(A3, newton_eigenpair, num_trials=15, seed=11, scale=2.0)
print_trial_rows(newton_rows_A3)


# Broyden's Method Extension

Now time to electric-boogaloo it for Broyden's method.

Broyden's method is a quasi-Newton method. Instead of recomputing the exact Jacobian at every iteration, it maintains an approximation `B_k` to the Jacobian. The step still solves

$$B_k s_k = -f(x_k),$$

but after taking the step, Broyden updates `B_k` using the change in the function values.

For the version below, called **good Broyden's method**, define

$$y_k = f(x_{k+1}) - f(x_k).$$

The Jacobian approximation is updated by

$$
B_{k+1}
= B_k + \frac{(y_k - B_k s_k)s_k^T}{s_k^T s_k}.
$$

This update forces the new approximation to satisfy the secant equation

$$B_{k+1}s_k = y_k,$$

which is the multidimensional analogue of the one-dimensional secant method.


In [ ]:
def broyden_eigenpair(A, x0, tol=1e-12, max_iter=50, verbose=True, initial_B="jacobian"):
    """Good Broyden's method for f(x)=0.

    Parameters
    ----------
    A : square NumPy array
        Symmetric matrix.
    x0 : NumPy array of length n+1
        Initial guess [v0; lambda0].
    tol : float
        Stop when ||f(x)||_2 < tol.
    max_iter : int
        Maximum number of Broyden steps.
    verbose : bool
        Print iteration table.
    initial_B : {"jacobian", "identity"}
        Starting approximation for the Jacobian.
        "jacobian" usually behaves more like Newton at first.
        "identity" is more generic but often needs more iterations.
    """
    x = np.array(x0, dtype=float).copy()
    m = len(x)

    if initial_B == "jacobian":
        B = J_eigenpair(x, A)
    elif initial_B == "identity":
        B = np.eye(m)
    else:
        raise ValueError("initial_B must be 'jacobian' or 'identity'")

    residuals = []

    if verbose:
        print(f"{'k':>3} | {'lambda':>15} | {'||f(x)||_2':>14} | {'step norm':>14}")
        print("-" * 58)

    step_norm = np.nan
    for k in range(max_iter + 1):
        fx = f_eigenpair(x, A)
        res = np.linalg.norm(fx)
        residuals.append(res)
        v, lam = split_x(x)

        if verbose:
            step_text = "---" if k == 0 else f"{step_norm:.6e}"
            print(f"{k:3d} | {lam:15.10f} | {res:14.6e} | {step_text:>14}")

        if res < tol:
            return x, np.array(residuals), True

        try:
            step = np.linalg.solve(B, -fx)
        except np.linalg.LinAlgError:
            if verbose:
                print("Broyden matrix B is singular or numerically singular.")
            return x, np.array(residuals), False

        x_next = x + step
        fx_next = f_eigenpair(x_next, A)
        y = fx_next - fx

        Bs = B @ step
        denom = step @ step
        if denom == 0 or not np.isfinite(denom):
            if verbose:
                print("Step became zero or invalid before convergence.")
            return x, np.array(residuals), False

        B = B + np.outer(y - Bs, step) / denom
        step_norm = np.linalg.norm(step)
        x = x_next

    return x, np.array(residuals), False


## 9. Broyden on the required 2 by 2 matrix

Run the same initial guess used for Newton. Try changing `initial_B` from `"jacobian"` to `"identity"` and watch how the convergence changes.


In [ ]:
# Same x0 as the Newton 2 by 2 experiment.
x_broyden_A2, broyden_residuals_A2, broyden_converged_A2 = broyden_eigenpair(
    A2, x0, tol=1e-13, max_iter=40, initial_B="jacobian"
)
v_broyden_A2, lam_broyden_A2 = split_x(x_broyden_A2)

print("\nConverged:", broyden_converged_A2)
print("lambda ≈", lam_broyden_A2)
print("v ≈", normalize_sign(v_broyden_A2))
print("||v||_2 ≈", np.linalg.norm(v_broyden_A2))
print("final residual ≈", broyden_residuals_A2[-1])


## 10. Compare Newton and Broyden residual plots for the 2 by 2 problem

Newton usually needs fewer iterations because it uses the exact Jacobian every time. Broyden often needs more iterations, but each iteration avoids recomputing the exact Jacobian.


In [ ]:
plt.figure()
plt.semilogy(residuals, marker="o", label="Newton")
plt.semilogy(broyden_residuals_A2, marker="s", label="Broyden")
plt.xlabel("Iteration k")
plt.ylabel("||f(x_k)||_2")
plt.title("Newton vs. Broyden residuals: 2 by 2 example")
plt.grid(True)
plt.legend()
plt.show()

print("Newton iterations:", len(residuals) - 1)
print("Broyden iterations:", len(broyden_residuals_A2) - 1)


## 11. Broyden random trials for the 2 by 2 matrix

This repeats the random-start experiment using Broyden's method. Compare the iteration counts and final residuals with the Newton random trials.


In [ ]:
broyden_rows_A2 = run_random_trials(
    A2,
    broyden_eigenpair,
    num_trials=12,
    seed=4,
    scale=2.0,
    tol=1e-12,
    max_iter=60,
    initial_B="jacobian",
)
print_trial_rows(broyden_rows_A2)


## 12. Estimate Broyden's convergence rate

Broyden's method is usually superlinear near a regular root when it behaves well, but it is not usually quadratically convergent like Newton's method. The residual-based estimate below is noisy, so treat it as a diagnostic rather than a theorem.


In [ ]:
print("Estimated orders from the earlier Broyden 2 by 2 run:")
for k, p in estimate_orders(broyden_residuals_A2):
    print(f"using residuals around k={k}: p ≈ {p:.3f}")


## 13. Broyden on the 3 by 3 symmetric example

Now apply Broyden's method to the same 3 by 3 matrix and starting vector used in the Newton section.


In [ ]:
x_broyden_A3, broyden_residuals_A3, broyden_converged_A3 = broyden_eigenpair(
    A3, x0_3, tol=1e-13, max_iter=60, initial_B="jacobian"
)
v_broyden_A3, lam_broyden_A3 = split_x(x_broyden_A3)

print("Converged:", broyden_converged_A3)
print("lambda ≈", lam_broyden_A3)
print("v ≈", normalize_sign(v_broyden_A3))
print("||v||_2 ≈", np.linalg.norm(v_broyden_A3))
print("final residual ≈", broyden_residuals_A3[-1])


## 14. Compare Newton and Broyden residual plots for the 3 by 3 problem

This comparison is often more interesting than the 2 by 2 case because there are more possible eigenpairs and more directions for the nonlinear iteration to move.


In [ ]:
plt.figure()
plt.semilogy(residuals3, marker="o", label="Newton")
plt.semilogy(broyden_residuals_A3, marker="s", label="Broyden")
plt.xlabel("Iteration k")
plt.ylabel("||f(x_k)||_2")
plt.title("Newton vs. Broyden residuals: 3 by 3 example")
plt.grid(True)
plt.legend()
plt.show()

print("Newton iterations:", len(residuals3) - 1)
print("Broyden iterations:", len(broyden_residuals_A3) - 1)


## 15. Broyden random trials for the 3 by 3 matrix

This repeats the 3 by 3 random-start experiment with Broyden's method.


In [ ]:
broyden_rows_A3 = run_random_trials(
    A3,
    broyden_eigenpair,
    num_trials=15,
    seed=11,
    scale=2.0,
    tol=1e-12,
    max_iter=80,
    initial_B="jacobian",
)
print_trial_rows(broyden_rows_A3)


## 16. Interactive experiment

The cell below gives sliders for the initial guess in the 2 by 2 problem. If widgets are unavailable, simply skip this cell and edit the earlier code cells instead.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def experiment(v1=0.8, v2=1.3, lam0=1.5, method="Newton", initial_B="jacobian"):
        x_start = np.array([v1, v2, lam0], dtype=float)
        if method == "Newton":
            x_out, res_hist, ok = newton_eigenpair(A2, x_start, tol=1e-11, max_iter=20, verbose=False)
        else:
            x_out, res_hist, ok = broyden_eigenpair(
                A2, x_start, tol=1e-11, max_iter=40, verbose=False, initial_B=initial_B
            )
        v, lam = split_x(x_out)
        print("initial x0 =", x_start)
        print("method =", method)
        print("converged =", ok)
        print("iterations =", len(res_hist) - 1)
        print("lambda ≈", lam)
        print("v ≈", normalize_sign(v))
        print("final residual ≈", res_hist[-1])

    display(widgets.interactive(
        experiment,
        v1=widgets.FloatSlider(value=0.8, min=-3.0, max=3.0, step=0.1),
        v2=widgets.FloatSlider(value=1.3, min=-3.0, max=3.0, step=0.1),
        lam0=widgets.FloatSlider(value=1.5, min=-2.0, max=6.0, step=0.1),
        method=widgets.Dropdown(options=["Newton", "Broyden"], value="Newton"),
        initial_B=widgets.Dropdown(options=["jacobian", "identity"], value="jacobian"),
    ))



## Congratulations

Thats it, the slogan that we didn't do it because its easy, but because we thought it was extremely, heavily, flashbacks of 3x the expected time spend proves that this applies here. I remise for any reader not mandated to be here. Please reflect on the journey that somehow ended here and wipe your memory, man and AI botscraper both.   
